In [1]:
import numpy as np
import pandas as pd
from numpy.linalg import matrix_rank
from numpy.linalg import inv
from numpy.linalg import det
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [2]:
def matrixSolver(X, y, x_ncol, y_ncol, l, bias):
    X = X.reshape(-1, x_ncol)
    # add bias column
    if bias == True:
        X = np.insert(X,0,1,axis=1)
    y = y.reshape(-1, y_ncol)
    n = X.shape[0]
    d = X.shape[1]
    # use ridge regression
    if l > 0:
        if d <= n: # col <= row, features <= eqns
            I = np.eye(d)
            w = inv(X.T@X+l*I)@X.T@y
            print(f"Ridge regression (PRIMAL form)\nw is\n{w}\nMSE is {mean_squared_error(y, X@w, multioutput='raw_values')}")
        else:
            I = np.eye(n)
            w = X.T@inv(X@X.T+l*I)@y
            print(f"Ridge regression (DUAL form)\nw is\n{w}\nMSE is {mean_squared_error(y, X@w, multioutput='raw_values')}")
        return w
    # uses ordinary least square
    if n > d:
        if not np.isclose(np.linalg.det(X.T@X), 0):
            w = inv(X.T@X)@X.T@y
            print(f"Over-determined({X.shape[0]}x{X.shape[1]}) system")
            print(f"det(X.T@X) = {np.linalg.det(X.T@X)} and w is\n{w}")
            print(f"MSE is {mean_squared_error(y, X@w, multioutput='raw_values')}")
        else:
            print("Over-determined system where det(X.T@X) = 0")
            return
    elif n < d:
        if not np.isclose(np.linalg.det(X@X.T), 0):
            w = X.T@inv(X@X.T)@y
            print(f"Under-determined({X.shape[0]}x{X.shape[1]}) system")
            print(f"det(X@X.T) = {np.linalg.det(X@X.T)} and w is\n{w}")
            print(f"MSE is {mean_squared_error(y, X@w, multioutput='raw_values')}")
        else:
            print("Under-determined system where det(X@X.T) = 0")
            return
    else:
        if not np.isclose(np.linalg.det(X), 0):
            w = inv(X)@y
            print(f"Even-determined({X.shape[0]}x{X.shape[1]}) system\ndet(X) = {np.linalg.det(X)} and w is\n{w}\nMSE is {mean_squared_error(y, X@w, multioutput='raw_values')}")
        else:
            print("Even-determined system where det(X) = 0")
            return
    return w

def polynomialRegression(X, y, x_ncol, y_ncol, deg, lamda):
    X = X.reshape(-1, x_ncol)
    y = y.reshape(-1, y_ncol)
    
    poly_features = PolynomialFeatures(degree=deg)
    P = poly_features.fit_transform(X)
    
    w = matrixSolver(P, y, P.shape[1], y_ncol, lamda, False)
    return w, P

def oneHotEncoder(y, y_ncol):
    y = y.reshape(-1,y_ncol)
    encoder = OneHotEncoder(sparse_output=False)
    y = encoder.fit_transform(y)
    return y

In [20]:
num = 0
denom = 0

for i in range(1,7):
    for j in range(1,7):
        curr = i + j
        if not curr%2==0 and curr>8:
            num += 1
        denom += 1

print(f"{num}/{denom}")

6/36


In [5]:
X = np.array([1,4,2,7,-3,11])
y = np.array([1,-2.5,4])
X = X.reshape(-1,2)
y = y.reshape(-1,1)
k = np.concatenate((X, y), axis=1)
print(k)
print(matrix_rank(X))
print(matrix_rank(k))

[[ 1.   4.   1. ]
 [ 2.   7.  -2.5]
 [-3.  11.   4. ]]
2
3


In [6]:
num = 0
denom = 0

for i in range(1,7):
    for j in range(1,7):
        curr = i + j
        if curr%2==0 and curr>8:
            num += 1
        denom += 1

print(f"{num}/{denom}")

4/36


In [9]:
y = np.array([9,3,6,7,5,6,3,1,2,9])
X = np.array([50,10,40,7,65,12,70,5,75,4])
w = matrixSolver(X,y,2,2,0,True)
X_test = np.array([[1,62,8]])
y_pred = X_test@w
print(y_pred)

Over-determined(5x3) system
det(X.T@X) = 167600.0000000005 and w is
[[ 9.14677804e+00  6.49701671e+00]
 [-1.16348449e-01 -4.95226730e-03]
 [ 3.72911695e-01 -1.31563246e-01]]
MSE is [1.22673031 8.01760143]
[[4.91646778 5.13747017]]


In [13]:
X = np.array([1,4,2,-3,10,5])
X = X.reshape(-1,3)
det(X@X.T)

np.float64(604.9999999999997)

In [19]:
X = np.array([3,3.5,9,2.3,7,5,2,7])
X = X.reshape(-1,2)
poly_features = PolynomialFeatures(degree=2)
p = poly_features.fit_transform(X)
print(p)
y = np.array([-1,1,-1,-1])
w, P = polynomialRegression(X,y,2,1,2,0)
X_test = np.array([[10,4.5]])
p = poly_features.fit_transform(X_test)
y_pred = p@w
print(y_pred)
X = np.array([1,1,0,0])
X = X.reshape(-1,2)
print(det(X))

[[ 1.    3.    3.5   9.   10.5  12.25]
 [ 1.    9.    2.3  81.   20.7   5.29]
 [ 1.    7.    5.   49.   35.   25.  ]
 [ 1.    2.    7.    4.   14.   49.  ]]
Under-determined(4x6) system
det(X@X.T) = 19156287579.566345 and w is
[[-0.07857695]
 [-0.15994819]
 [-0.16861362]
 [ 0.04830985]
 [-0.0546574 ]
 [ 0.02348434]]
MSE is [1.33518406e-27]
[[0.41014004]]
0.0


In [9]:
X = np.array([71,3,15,59,7,12,65,5,9,81,3,19,55,6,11])
y = np.array([5,7,10,8,3,8,7,3,6,7])
w = matrixSolver(X,y,3,2,0,True)
X_test = np.array([[75,5,11]])
poly_features = PolynomialFeatures(degree=1)
p = poly_features.fit_transform(X_test)
y_pred = p@w
print(y_pred)

Over-determined(5x4) system
det(X.T@X) = 111239.99999998909 and w is
[[-1.34796117e+01  1.73277778e+01]
 [-1.94174757e-03 -7.22222222e-02]
 [ 1.88932039e+00 -2.00000000e-01]
 [ 8.13592233e-01 -3.77777778e-01]]
MSE is [0.33553398 0.68666667]
[[4.77087379 6.75555556]]


In [10]:
X = np.array([4,-5,0.3,1,12,-3])
X = X.reshape(-1,3)
print(det(X@X.T))

3090.2499999999995


In [15]:
X = np.array([50,10,40,7,65,12,70,5,75,4])
y = np.array([9,3,6,7,5,6,3,1,2,9])
w = matrixSolver(X,y,2,2,0,True)

X_test = np.array([[1,42,8]])
y_pred = X_test@w
print(y_pred)

X_test = np.array([[42,8]])
w, P = polynomialRegression(X, y, 2, 2, 3, 0)
poly_features = PolynomialFeatures(degree=3)
K = poly_features.fit_transform(X_test)
y_pred = K@w
print(y_pred)

Over-determined(5x3) system
det(X.T@X) = 167600.0000000005 and w is
[[ 9.14677804e+00  6.49701671e+00]
 [-1.16348449e-01 -4.95226730e-03]
 [ 3.72911695e-01 -1.31563246e-01]]
MSE is [1.22673031 8.01760143]
[[7.24343675 5.23651551]]
Under-determined(5x10) system
det(X@X.T) = 8.578729748155386e+34 and w is
[[-2.03249134e-04  1.17582184e-03]
 [-3.35069997e-03  2.04338138e-02]
 [-4.56871363e-04  2.90502031e-03]
 [ 8.16522849e-03  7.75694847e-03]
 [ 2.25429984e-03 -1.95845623e-03]
 [ 2.55823956e-03 -1.17651157e-02]
 [-1.12475764e-04  1.75965833e-04]
 [ 5.64581115e-04 -8.22276065e-03]
 [-1.00388844e-02  7.34135423e-02]
 [ 3.75156263e-02 -1.98782973e-01]]
MSE is [5.38094672e-17 9.14647528e-16]
[[7.03779131 5.71096267]]


In [19]:
X = np.array([1,4,2,-3,10,5])
X = X.reshape(-1,3)
print(det(X.T@X))
print(det(X@X.T))

0.0
604.9999999999997


In [20]:
X = np.array([2,0,0,0,-1,1])
X = X.reshape(-1,3)
print(det(X.T@X))

0.0


In [24]:
num = 0
denom = 0
for i in range(1,7):
    for j in range(1,7):
        for k in range(1,7):
            denom += 1
            tot = i+j+k
            if tot % 2 != 0 and tot <= 5:
                num += 1
print(num/denom)
print(f"{num}/{denom}")

0.032407407407407406
7/216


In [36]:
X = np.array([3,4,5,6,7])
y = np.array([5,4,3,2,1])
X = X.reshape(-1,1)
y = y.reshape(-1,1)
w = matrixSolver(X,y,1,1,0,True)
X_test = np.array([[1,8]])
y_pred = X_test@w
print(y_pred)

Over-determined(5x2) system
det(X.T@X) = 49.99999999999999 and w is
[[ 8.]
 [-1.]]
MSE is 2.871453695004483e-29
[[6.21724894e-15]]


In [40]:
X = np.array([1,2,0,6,1,0,0,5,1,7])
y = np.array([1,2,3,4,5])
X = X.reshape(-1,2)
y = y.reshape(-1,1)
w = matrixSolver(X,y,2,1,0,True)
X_test = np.array([[1,1,3]])
y_pred = X_test@w
print(y_pred)

Over-determined(5x3) system
det(X.T@X) = 158.99999999999986 and w is
[[1.13207547]
 [0.8490566 ]
 [0.33962264]]
MSE is 1.388679245283019
[[3.]]


In [41]:
X = np.array([1,2,2.9])
y = np.array([2,4.1,6.1])
w = matrixSolver(X,y,1,1,0,True)

Over-determined(3x2) system
det(X.T@X) = 5.419999999999997 and w is
[[-0.17509225]
 [ 2.15682657]]
MSE is 0.0007441574415744267


In [42]:
X = np.array([1,2,2.9])
y = np.array([2,4.1,6.1])
w = matrixSolver(X,y,1,1,0.1,True)
X_test = np.array([[1,1.5]])
y_pred = X_test@w
print(y_pred)

Ridge regression (PRIMAL form)
w is
[[0.03832556]
 [2.04765945]]
MSE is 0.007922748024973668
[[3.10981474]]


In [ ]:
X = np.array([1,2,3,4,0,6,1,1,0,0,1,2,5,7,-2,-1,4,0])
y = np.array([1,1,2,3,2,3])
y = oneHotEncoder(y, 1)
X_test = np.array([[1,-2,3]])
w, P = polynomialRegression(X,y,3,3,3,0)
poly_features = PolynomialFeatures(degree=3)
P = poly_features.fit_transform(X_test)
print(P)
y_pred = P@w
print(y_pred)
print(np.argmax(P @ w, axis=1)) # returns the index of the maximum value along each column(axis=1)

[[1. 0. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [0. 0. 1.]]
Over-determined(6x4) system
det(X.T@X) = 57646.99999999992 and w is
[[-0.2424931   0.92743768  0.31505542]
 [ 0.01542144  0.18137978 -0.19680122]
 [ 0.09029091 -0.19338387  0.10309296]
 [ 0.21626451 -0.2752962   0.05903169]]
MSE is 0.04854449397964237
[[0.24114004 0.6696966  0.08916336]]


In [ ]:
X = np.array([1,2,3,4,0,6,1,1,0,0,1,2,5,7,-2,-1,4,0])
y = np.array([1,1,2,3,2,3])
y = oneHotEncoder(y, 1)
X_test = np.array([[1,-2,3]])
w, P = polynomialRegression(X,y,3,3,3,0)
poly_features = PolynomialFeatures(degree=3)
P = poly_features.fit_transform(X_test)
print(P)
y_pred = P@w
print(y_pred)
print(np.argmax(P @ w, axis=1))

Under-determined(6x20) system
det(X@X.T) = 5.047578389024734e+18 and w is
[[-0.01176815  0.17482347  0.04418372]
 [ 0.00193766  0.16820103 -0.0107976 ]
 [-0.00746191  0.17015972  0.03168837]
 [-0.01416555 -0.00098195  0.06758816]
 [-0.00031705  0.15536443 -0.00719315]
 [ 0.0069067   0.1410684  -0.02408252]
 [ 0.01156987  0.00117843 -0.04089203]
 [-0.00069248  0.14315417  0.0118072 ]
 [ 0.00115565  0.00040379  0.02366433]
 [-0.0171958  -0.01415357  0.09575615]
 [-0.00836554  0.08163637  0.00475135]
 [ 0.00236418  0.04311679 -0.00934772]
 [ 0.00364828  0.03395263 -0.03727903]
 [ 0.01310783 -0.02877387 -0.04008716]
 [ 0.03027174  0.03744565 -0.0904618 ]
 [ 0.02461518 -0.03053105 -0.11170377]
 [ 0.00428875 -0.04875049 -0.00092249]
 [ 0.03176538  0.04824686 -0.06794225]
 [ 0.02954106 -0.05532353 -0.03283009]
 [-0.00890728 -0.03210198  0.07686481]]
MSE is 2.4387047504695385e-29
[[  1.   1.  -2.   3.   1.  -2.   3.   4.  -6.   9.   1.  -2.   3.   4.
   -6.   9.  -8.  12. -18.  27.]]
[[-0.5164

In [78]:
X = np.array([1,3,2,3,6,4,5,-1,6,9,8,0])
y = np.array([1,3,9,3])
X = X.reshape(-1,3)
y= y.reshape(-1,1)
print(X)
print(y)
print(matrix_rank(X))
D = np.hstack((X,y))
print(D)
print(matrix_rank(D))
matrixSolver(X,y,3,1,0,False)

[[ 1  3  2]
 [ 3  6  4]
 [ 5 -1  6]
 [ 9  8  0]]
[[1]
 [3]
 [9]
 [3]]
3
[[ 1  3  2  1]
 [ 3  6  4  3]
 [ 5 -1  6  9]
 [ 9  8  0  3]]
4
Over-determined(4x3) system
det(X.T@X) = 186975.99999999977 and w is
[[ 0.71127845]
 [-0.4244395 ]
 [ 0.83732672]]
MSE is 0.004193051514632901


array([[ 0.71127845],
       [-0.4244395 ],
       [ 0.83732672]])

In [70]:
X = np.array([2,0,0,0,-1,1])
X = X.reshape(-1,3)
det(X.T@X)

np.float64(0.0)